# Snowflake Notebook上でのデータ参照
**SQLの場合**
```sql
USE Database <データベース名>;
USE SCHEMA <スキーマ名>;
-- テーブル名を絶対参照する
SELECT ~ FROM [データベース名.][スキーマ名.]<テーブル名>;
```

**Pythonの場合**
```python
from snowflake.snowpark.cotext import get_active_session

session = get_active_session()
session.use_database('<データベース名')
session.use_schema('<スキーマ名>')
df = session.table('[データベース名.][スキーマ名.]<テーブル名>')
df = session.sql('<SQL クエリ>')
```

In [ ]:
USE DATABASE SNOWFLAKE_SAMPLE_DATA;
USE SCHEMA TPCH_SF1;
-- サンプルデータベースをSQLで参照してみる
SELECT * FROM TPCH_SF1.CUSTOMER limit 10;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
# session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER").sample(n=10)
session.table("CUSTOMER").sample(n=10)

# Snowflake上でのPandasを使ったデータエンジニアリング
## Avalanche (架空のウィンタースポーツ用品会社)

Avalancheの注文履歴・出荷データを, [Snowflake 上で動作する pandas](https://docs.snowflake.com/en/developer-guide/snowpark/python/pandas-on-snowflake) を使って分析します

In [ ]:
# Snowpark Pandas API
import modin.pandas as spd
# Import the Snowpark pandas plugin for modin
import snowflake.snowpark.modin.plugin
import streamlit as st

import snowflake.snowpark.functions as F
from snowflake.snowpark.context import get_active_session

In [ ]:
# Snowflake のアクティブなセッション（現在接続中のセッション）を取得する
# これによって、以降の処理で Snowflake に対して SQL 実行やデータ操作ができるようになる
session = get_active_session()

# セッションに「クエリタグ (query tag)」を設定する
# クエリタグとは、Snowflake 上で実行した SQL クエリに「ラベル」を付ける仕組み
# これにより、モニタリングやトラブルシューティングで
# 「どのアプリから来たクエリか」「どのハンズオン教材からの実行か」などを追跡できる
session.query_tag = {
    "origin": "sf_devrel",          # クエリの発行元（ここでは Snowflake Developer Relations の意味）
    "name": "de_100_vhol",          # このハンズオンや演習の名前
    "version": {                    # バージョン情報
        "major": 1,
        "minor": 0
    },
    "attributes": {                 # 追加の属性情報（カスタムラベルのようなもの）
        "is_quickstart": 1,         # Quickstart チュートリアルからの実行であることを示す
        "source": "notebook",       # Jupyter Notebook や Snowflake Notebook からの実行であることを示す
        "vignette": "snowpark_pandas"  # この教材のシナリオ名（Snowpark + pandas のハンズオンであること）
    }
}

session.use_database("AVALANCHE_DB")
session.use_schema("AVALANCHE_SCHEMA")

session

### TODO: ダウンロードした, 出荷データ(shipping-logs.csv)を Notebooks ワークスペースに読み込む
- 画面左側の[➕]ボタンからファイルをアップロード

In [ ]:
# Snowpark pandas（spd）を使って CSV ファイルを読み込む
# 'shipping-logs.csv' という名前のCSVファイルを対象にしている
# CSVの中に 'shipping_date' という列があり、それを日付型（datetime型）として扱うよう指定している
shipping_logs_mdf = spd.read_csv(
    'shipping-logs.csv',        # 読み込むCSVファイルの名前
    parse_dates=['Shipping Date']  # この列を「文字列」ではなく「日付」として読み込む
)

# 読み込んだデータ（shipping_logs_mdf）を表示する
# shipping_logs_mdf は pandas.DataFrame と同じように扱えるオブジェクト
shipping_logs_mdf


### TODO: ダウンロードした, 注文履歴データ(order-history.csv)を Notebooks ワークスペースに読み込む
- 画面左側の[➕]ボタンからファイルをアップロード

### TODO: ワークスペースに読み込んだファイルをステージに保存

In [ ]:
-- ファイル格納先の内部ステージを作成
CREATE OR REPLACE STAGE data_file
    DIRECTORY = ( ENABLE = TRUE
                  AUTO_REFRESH = TRUE
                  );

In [ ]:
session = get_active_session()
put_result = session.file.put("order-history.csv","@data_file", auto_compress= False)

Stageに格納したcsvファイルをSnowpark Pandasを使って読み込む

```spd.read_csv(<file_path_in_stage>)```

`@[データベース名.][スキーマ名.]<stage_name>/ファイルパス`

In [ ]:
# Snowpark pandas（spd）を使って ステージから CSV ファイルを読み込む
# 'order-history.csv' という名前のCSVファイルを対象にしている
order_history_mdf = spd.read_csv(
    '@data_file/order-history.csv'   # 読み込むCSVファイルへのパス
)

# 読み込んだデータ（order_history_mdf）を表示する
# order_history_mdf は pandas.DataFrame と同じように扱えるオブジェクト
order_history_mdf


In [ ]:
# order_history_mdf の列名を分かりやすく変更する
# rename(columns={...}) で「元の列名 : 新しい列名」を指定する

order_history_mdf = order_history_mdf.rename(columns = {
    'Order ID': 'order_id',              # 注文ID → order_id
    'Customer ID': 'customer_id',        # 顧客ID → customer_id
    'Product ID': 'product_id',          # 商品ID → product_id
    'Product Name': 'product_name',      # 商品名 → product_name
    'Quantity Ordered': 'quantity_ordered',  # 注文数 → quantity_ordered
    'Price': 'price',                    # 単価 → price
    'Total Price': 'total_price',        # 合計金額 → total_price
    'Ordered Date': 'date'                       # 日付 → date
})

# 列名が正しく変更されたかを確認する
order_history_mdf.columns

### 価格カラムから $ 記号を取り除いて整理する

In [ ]:
# 文字列で表現された価格（例: "$19.99"）を数値に変換する関数
def clean_price(price_str):
    # 価格の文字列から "$" 記号を取り除き、前後の余分な空白も削除する
    # 例: " $19.99 " → "19.99"
    cleaned = price_str.replace('$', '').strip()
    
    # 文字列になっている数値を float型（小数点を持つ数値）に変換する
    # 例: "19.99" → 19.99
    return float(cleaned)


In [ ]:
# ---- 価格カラムを数値に変換する処理 ----

# 'price' 列に対して clean_price 関数を適用する
# これにより "$19.99" のような文字列が 19.99 という float型の数値になる
order_history_mdf['price'] = order_history_mdf['price'].apply(clean_price)

# 'total_price' 列に対しても同じく clean_price を適用する
# これで合計金額も数値として扱えるようになる
order_history_mdf['total_price'] = order_history_mdf['total_price'].apply(clean_price)


# ---- 変換後のデータ型を確認する処理 ----

# price 列のデータ型を表示（float になっていればOK）
print("\nPrice column data type:", order_history_mdf['price'].dtype)

# total_price 列のデータ型を表示（こちらも float になっていればOK）
print("Total price column data type:", order_history_mdf['total_price'].dtype)


In [ ]:
# 実際に $ が消えて数値化されているかを表で確認
order_history_mdf.head()



### 製品ごとの注文数を計算する：order_history と shipping_logs を結合する

In [ ]:
# order_history_mdf の列名を分かりやすく変更する
# rename(columns={...}) で「元の列名 : 新しい列名」を指定する

shipping_logs_mdf = shipping_logs_mdf.rename(columns = {
    'Order ID': 'order_id',              # 注文ID → order_id
    'Shipping Date': 'shipping_date',    # 発送日 → shipping_date
    'Carrier': 'carrier',                # 配送業者 → carrier
    'Tracking Number': 'trucking_number',# 追跡番号 → trucking_nrumber
    'Latitude': 'latitude',              # 緯度 → lattitude
    'Longitude': 'longitude',            # 経度 → longitude
    'Shipping Status': 'status'         # ステータス → status
})

# 列名が正しく変更されたかを確認する
shipping_logs_mdf.columns

In [ ]:
# ---- 注文データと出荷データを結合 ----

# order_history_mdf（注文データ）と shipping_logs_mdf（出荷データ）を
# 'order_id' 列をキーにして結合（マージ）する

order_shipping_mdf = spd.merge(
    order_history_mdf,      # 左側のデータフレーム（注文履歴）
    shipping_logs_mdf,      # 右側のデータフレーム（出荷ログ）
    on='order_id',          # 結合キーとなる列
    how='inner'             # 内部結合（両方に存在するデータのみ）
)

# 結合後のデータフレームの先頭5行を表示して確認
order_shipping_mdf.head(5)



In [ ]:
# ---- 商品ごとの注文件数を集計 ----

# 'product_name' 列でグループ化して、注文件数を数える
# reset_index(name='order_count') で結果をデータフレーム形式に戻し、列名を 'order_count' に設定
product_counts_mdf = order_shipping_mdf.groupby('product_name').size().reset_index(name='order_count')

# ---- 注文件数の多い順に並べ替え ----

# sort_values() で 'order_count' 列を降順（ascending=False）に並べる
product_counts_mdf = product_counts_mdf.sort_values('order_count', ascending=False)

# ---- 結果を表示 ----
print("\nProduct Order Counts:")
st.dataframe(product_counts_mdf)



### 注文の配送ステータスごとにピボットする

In [ ]:
# ---- 商品ごとの注文ステータス別集計 ----

# **** を使って集計
# index='product_name' → 行に商品名を設定
# columns='status' → 列に注文ステータス（例: shipped, pending, cancelled）を設定
# values='order_id' → 注文IDを数える対象にする
# aggfunc='count' → 各セルに注文数をカウント
# fill_value=0 → データがない場合は 0 を埋める
product_status_pivot_mdf = order_shipping_mdf.pivot_table(
    index='product_name',
    columns='status',
    values='order_id',
    aggfunc='count',
    fill_value=0
)

# ---- 合計注文数の列を追加 ----

# 行ごとの合計を計算して 'Total_Orders' 列として追加
product_status_pivot_mdf['Total_Orders'] = product_status_pivot_mdf.sum(axis=1)

# ---- 合計注文数の多い順に並べ替え ----

product_status_pivot_mdf = product_status_pivot_mdf.sort_values('Total_Orders', ascending=False)

# ---- 結果を表示 ----
print("\nProduct Orders by Status:")
st.dataframe(product_status_pivot_mdf)


## Avalanche社は、各製品に対する顧客レビューについても理解したいと考えています。  


この分析を [Snowpark DataFrame API](https://docs.snowflake.com/en/developer-guide/snowpark/python/working-with-dataframes) を使って実行してみましょう。

In [ ]:
-- ---- データベースとスキーマの作成（Snowsight UI で実行する場合） ----
-- CREATE OR REPLACE DATABASE avalanche_db;
-- CREATE OR REPLACE SCHEMA avalanche_schema;

-- 既存データベースを使用する
USE DATABASE avalanche_db;

-- 既存スキーマを使用する
USE SCHEMA avalanche_schema;


-- ---- ファイルを格納するステージ（Stage）の作成 ----
-- Stage とは、Snowflake にデータを取り込む前に一時的にファイルを置いておく場所です
CREATE OR REPLACE STAGE avalanche_stage
  URL = 's3://sfquickstarts/misc/avalanche/csv/'  -- S3 バケットの場所を指定
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE); -- ディレクトリ構造を有効化、自動更新ON

-- Stage 内のファイル一覧を確認
ls @avalanche_stage;


### 顧客レビューを Snowflake のテーブルに読み込む

In [ ]:
-- ---- テーブルの作成 ----
-- customer_reviews という名前のテーブルを作成
-- 商品名、レビュー日、レビュー本文、感情スコアを保存する
CREATE OR REPLACE TABLE customer_reviews (
    product VARCHAR,          -- 商品名（文字列）
    date DATE,                -- レビュー日（DATE型）
    summary TEXT,             -- レビュー本文（TEXT型）
    sentiment_score FLOAT     -- 感情スコア（数値、小数点）
);


-- ---- CSV ファイルからデータをテーブルにロード ----
COPY INTO customer_reviews
FROM @avalanche_stage/customer_reviews.csv   -- 先ほど作成した Stage 内の CSV を指定
FILE_FORMAT = (
    TYPE = CSV,                               -- CSV形式のファイル
    FIELD_DELIMITER = ',',                     -- カラム区切り文字はカンマ
    SKIP_HEADER = 1,                           -- 1行目はヘッダーなのでスキップ
    FIELD_OPTIONALLY_ENCLOSED_BY = '"',       -- 値が " " で囲まれている場合に対応
    TRIM_SPACE = TRUE,                         -- 前後の空白を削除
    NULL_IF = ('NULL', 'null'),               -- "NULL" または "null" は NULL として扱う
    EMPTY_FIELD_AS_NULL = TRUE                -- 空欄も NULL として扱う
);


In [ ]:
# ---- Snowflake テーブルを Snowpark DataFrame として読み込む ----

# 'customer_reviews' テーブルを Snowpark DataFrame として取得
customer_reviews_sdf = session.table('customer_reviews')

# 取得した Snowpark DataFrame の内容を確認
customer_reviews_sdf


In [ ]:
# ---- Product単位でSENTIMENT_SCOREの平均を集計する ----
product_sentiment_sdf = customer_reviews_sdf.group_by('PRODUCT') \
    .agg(F.round(F.avg('SENTIMENT_SCORE'),2).alias('AVG_SENTIMENT_SCORE')) \
    .sort(F.col('AVG_SENTIMENT_SCORE').desc())

# Display the results
print("\nAverage Sentiment Scores by Product:")
product_sentiment_sdf

## Snowpark DataframeとPandas on Snowflakeの比較

In [ ]:
import pandas as pd
from time import perf_counter
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
# pandas on Snowflakeを利用した場合
start = perf_counter()
pd_df = spd.read_snowflake("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1000.CUSTOMER")
end = perf_counter()
data_size = len(pd_df)
print(f"Snowpark Pandasに {data_size:,}行のデータを読み込み。処理時間 {(end - start):,} 秒")

In [ ]:
# DataFrameをpandas on Snowflakeに変換する
start = perf_counter()
pd_df = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1000.CUSTOMER").to_snowpark_pandas()
end = perf_counter()
data_size = len(pd_df)
print(f"Snowpark Pandasに {data_size:,}行のデータを読み込み。処理時間 {(end - start):,} 秒")

大量データをpandasを使ってメモリに取り込もうとするとOut of Memoryが発生する可能性があるため、Warehouseを`Snowpark最適化`に変更してサイズを拡張する

In [ ]:
ALTER WAREHOUSE NOTEBOOK_HO_WH SET 
    WAREHOUSE_TYPE = 'SNOWPARK-OPTIMIZED'
    WAREHOUSE_SIZE = MEDIUM;

In [ ]:
# Dataframe APIを利用して、Pandasにデータを格納した場合
start = perf_counter()
pd_pf = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1000.CUSTOMER").to_pandas()
end = perf_counter()
data_size = len(pd_pf)
print(f"pandasに {data_size:,}行のデータを読み込み。処理時間 {(end - start):,} 秒")

In [ ]:
ALTER WAREHOUSE NOTEBOOK_HO_WH SET 
    WAREHOUSE_TYPE = STANDARD
    WAREHOUSE_SIZE = XSMALL;

大規模データセットを扱う場合に`Dataframe`を直接Pandasに変換せず、[to_pandas_batches](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/snowpark/api/snowflake.snowpark.DataFrame.to_pandas_batches)を利用する方法もあります。

```
df = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1000.CUSTOMER")
for pd_df in df.to_pandas_batches:
  print(pd_df)
```

## 📊 データの可視化

[Altair](https://altair-viz.github.io/)を使用して、データ分布をヒストグラムとして簡単に可視化できます。

In [ ]:
import altair as alt

# 追加したパッケージ
import matplotlib.pyplot as plt

pdf = customer_reviews_sdf.to_pandas()
chart = alt.Chart(pdf, title='評価分布').mark_bar().encode(
    alt.X("SENTIMENT_SCORE", bin=alt.Bin(step=0.5)),
    y='count()'
)

st.altair_chart(chart)

チャートをカスタマイズして、カーネル密度推定（KDE）と中央値をプロットしたいとします。matplotlibを使用して価格分布をプロットできます。`.plot`コマンドは内部的に`scipy`を使用してKDEプロファイルを計算することに注意してください。これは、このチュートリアルの前半でパッケージとして追加したものです。

In [ ]:
price = order_history_mdf["price"]

In [ ]:
fig, ax = plt.subplots(figsize = (6,3))
plt.tick_params(left = False, right = False , labelleft = False) 

price = order_history_mdf["price"]
price.plot(kind = "hist", density = True, bins = 15)
price.plot(kind="kde", color='#c44e52')


# パーセンタイルを計算
median = price.median()
ax.axvline(median,0, color='#dd8452', ls='--')
ax.text(median,0.8, f'Median: {median:.2f}  ',
        ha='right', va='center', color='#dd8452', transform=ax.get_xaxis_transform())

# チャートを美しくする
plt.style.use("bmh")
plt.title("Price Distribution")
plt.xlabel("Price (Binned)")
left, right = plt.xlim()   
plt.xlim((0, right))  
# 目盛りと軸線を削除
ax.tick_params(left = False, bottom = False)
for ax, spine in ax.spines.items():
    spine.set_visible(False)

plt.show()

## サブクエリ/セル間の参照

セルに名前をつけて、後続のセルでその出力を参照することができます


Jinjaを利用して別のSQLセルからSQLテーブルを参照することで、CTEを簡素化することができます。

```sql
SELECT * FROM {{cell}}
```

In [ ]:
select 
    product, 
    avg(sentiment_score) as avg_score, 
    min(sentiment_score) as min_score, 
    max(sentiment_score) as max_score
from customer_reviews
group by all;

In [ ]:
select * from {{subqueries}}
WHERE avg_score > 0.5;

SQL結果にPythonから直接アクセスし、結果をpandas DataFrameに変換できます。🐼

```python
# SQLセルの出力をSnowpark DataFrameとしてアクセス
my_snowpark_df = sql_querying.to_df()
``` 

```python
# SQLセルの出力をpandas DataFrameに変換
my_df = sql_querying.to_pandas()
``` 

In [ ]:
my_df = subqueries2.to_df()
my_df

## ステージパッケージの追加
使用したいPythonパッケージがAnacondaで利用できない場合は、パッケージをステージにアップロードし、ステージからインポートすることができます。ここでは、カスタムパッケージをノートブックにインポートする簡単な例を示します。

In [ ]:
-- ステージの作成
CREATE OR REPLACE STAGE AVALANCHE_DB.AVALANCHE_SCHEMA.FILE DIRECTORY = (ENABLE = TRUE);

// Step3: 公開されているGitからスクリプトを取得 //
-- Git連携のため、API統合を作成する
CREATE OR REPLACE API INTEGRATION git_api_integration
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/sfc-gh-skawakami/')
  ENABLED = TRUE;

-- GIT統合の作成
CREATE OR REPLACE GIT REPOSITORY GIT_INTEGRATION_FOR_HANDSON
  API_INTEGRATION = git_api_integration
  ORIGIN = 'https://github.com/sfc-gh-skawakami/sfc-jp-notebook_de_101.git';


ALTER GIT REPOSITORY GIT_INTEGRATION_FOR_HANDSON FETCH;

-- チェックする
ls @GIT_INTEGRATION_FOR_HANDSON/branches/main;

-- Githubからファイルを持ってくる
COPY FILES INTO @AVALANCHE_DB.AVALANCHE_SCHEMA.FILE FROM @GIT_INTEGRATION_FOR_HANDSON/branches/main/simple.zip;
ls @AVALANCHE_DB.AVALANCHE_SCHEMA.FILE;

`simple.zip`の内容

simple/__init__.py

```python
import streamlit as st

def greeting():
  return "Hello world!"

def hi():
  st.write(greeting())
```

次に進むために、メニューの[パッケージ]を開き、ステージパッケージを選択します。
以下を入力します
```python
@AVALANCHE_DB.AVALANCHE_SCHEMA.FILE/simple.zip
```

In [ ]:
import simple

simple.hi()

## プライベートリポジトリからのインポート
```
コンテナランタイムでのみ使用可能
```
Pipは、JFrog Artifactoryのようなプライベートソースからのパッケージのインストールを基本認証でサポートしています。ノートブックを外部アクセス統合（External Access Integration）用に設定し、リポジトリにアクセスできるようにします。

1. ネットワークルールを作成し、アクセスしたいリポジトリを指定します。例えば、このネットワークルールはJFrogリポジトリを指定しています:
```sql
CREATE OR REPLACE NETWORK RULE jfrog_network_rule
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = ('<your-repo>.jfrog.io');
```
2. 外部ネットワーク位置への認証に必要な資格情報を表すシークレットを作成します。
```sql
CREATE OR REPLACE SECRET jfrog_token
  TYPE = GENERIC_STRING
  SECRET_STRING = '<your-jfrog-token>';
```
3. リポジトリへのアクセスを許可する外部アクセス統合を作成します：
```sql
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION jfrog_integration
  ALLOWED_NETWORK_RULES = (jfrog_network_rule)
  ALLOWED_AUTHENTICATION_SECRETS = (jfrog_token)
  ENABLED = TRUE;

GRANT USAGE ON INTEGRATION jfrog_integration TO ROLE data_scientist;
```
4. 外部アクセス統合とシークレットをノートブックに関連付けます。
```sql
ALTER NOTEBOOK my_notebook
  SET EXTERNAL_ACCESS_INTEGRATIONS = (jfrog_integration),
    SECRETS = ('jfrog_token' = jfrog_token);
```
5. 外部アクセス設定にアクセスするには、ノートブックの右上にある「ワークシートのその他のアクション」（ノートブックアクションメニュー）を選択します。
6. 「ノートブック設定」を選択し、次に「External Access」タブを選択します。
7. リポジトリに接続する外部アクセス統合を選択します。
ノートブックが再起動します。
8. ノートブックが再起動されたら、リポジトリからインストール可能になります
```cmd
!pip install hello-jfrog --index-url https://<user>:<token>@<your-repo>.jfrog.io/artifactory/api/pypi/test-pypi/simple
```

# Snowflake Notebooksでファイルを扱う方法 🗄️

この例では、notebooksでファイルを扱う方法と、それらをstageに永続的に保存する方法を説明します。

## 一時ファイルの操作

notebookから書き込んだファイルは、notebookに関連付けられたlocal stageに一時的に保存されます。

**notebookセッションを終了するとすぐに、これらのファイルにアクセスできなくなることに注意してください。**

簡単なファイルを作成して、この仕組みの例を見てみましょう。

In [ ]:
import os
os.mkdir("myfolder/")
os.chdir("myfolder/")

In [ ]:
with open("myfile.txt",'w') as f:
    f.write("abc")
f.close()

stageにあるファイルを確認してみましょう。`notebook_app.ipynb`と`environment.yml`はSnowflake notebookの一部として自動的に作成されるファイルです。新しく作成したファイル`myfile.txt`が確認できます。

In [ ]:
import os
os.listdir()

では、notebookをセッションから切断してみましょう。これは、ブラウザページを閉じる/更新するか、右上の`Active`ボタンをクリックして`セッションを終了`を押すことで行えます。

このセルから開始してnotebookを再実行すると、前回のnotebookセッション中に作成したファイル`myfile.txt`は失われます。 

In [ ]:
import os
os.listdir()

## 永続ファイルの操作

セッションに戻ったときに再度アクセスできる永続的な場所にファイルを保存したい場合はどうでしょうか？例えば、モデルを訓練して後で使用するためにモデルを保存したい場合や、分析結果を保存したい場合があります。notebookセッション中に作成されたファイルはデフォルトで一時的なものなので、永続的なSnowflake stageにファイルを移動してファイルを永続的に保存する方法を説明します。

まず、`PERMANENT_STAGE`という名前のstageを作成しましょう：

In [ ]:
CREATE OR REPLACE STAGE PERMANENT_STAGE;

では、再び一時的なlocal stageに`myfile.txt`を書き込みましょう

In [ ]:
with open("myfile.txt",'w') as f:
    f.write("abc")
f.close()

では、Snowparkを使用して作成したローカルファイルをstageの場所にアップロードしましょう。Notebooksでは、`get_active_session`メソッドを使用して[session](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.Session#snowflake.snowpark.Session)コンテキスト変数を取得し、以下のようにSnowparkを操作できます：

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

Snowparkの[session.file.put](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.FileOperation.put)コマンドを使用して、`myfile.txt`をstageの場所`@PERMANENT_STAGE`に移動しましょう

In [ ]:
put_result = session.file.put("myfile.txt","@PERMANENT_STAGE", auto_compress= False)
put_result[0].status

ファイルが永続stageにアップロードされました。 

In [ ]:
LS @PERMANENT_STAGE;

notebookセッションを切断しても、ファイルが永続stageに残存していることが確認できます。

In [ ]:
LS @PERMANENT_STAGE;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

f = session.file.get_stream("@PERMANENT_STAGE/myfile.txt")
print(f.readline())
f.close()

また、読み取る前にファイルをローカルにダウンロードしたい場合は、[session.file.get](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.FileOperation.get)コマンドを使用できます： 

In [ ]:
# Download the file from stage to current local path
get_status = session.file.get("@PERMANENT_STAGE/myfile.txt","./")
get_status[0].status

In [ ]:
import os
os.listdir()

## ボーナス: stageからのデータファイルの操作

stageは、Snowflakeに読み込まれる前にデータファイルを保存する一般的な場所です。前のセクションでは、Snowflake stageに汎用ファイルを読み書きする方法を見ました。ここでは、stageに保存されたテーブル形式のデータファイルを操作する一般的な例をいくつか紹介します。


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

異なる日における様々なスキーリゾート地での降雪量を記録したサンプルデータセットがあります。

In [ ]:
# Create a Snowpark DataFrame with sample data
df = session.create_dataframe([[1, 'Big Bear', 8],[2, 'Big Bear', 10],[3, 'Big Bear', 5],
                               [1, 'Tahoe', 3],[2, 'Tahoe', 20],[3, 'Tahoe', 13]], 
                              schema=["DAY", "LOCATION", "SNOWFALL"])
df

Snowpark dataframeをstage上のCSVファイルに書き込む方法は次の通りです：

In [ ]:
df.write.copy_into_location("@PERMANENT_STAGE/snowfall.csv",file_format_type="csv",header=True)

stage上のファイルにアクセスするには、stageの場所からCSVファイルを読み取ってSnowpark dataframeに戻します：

In [ ]:
df = session.read.options({"infer_schema":True}).csv('@PERMANENT_STAGE/snowfall.csv')

notebooksでデータファイルを操作する方法について詳しく学ぶには、[外部S3 stageからCSVファイルを操作する方法](https://github.com/Snowflake-Labs/snowflake-demo-notebooks/blob/main/Load%20CSV%20from%20S3/Load%20CSV%20from%20S3.ipynb)と[パブリックエンドポイントからSnowflakeテーブルにデータを読み込む方法](https://github.com/Snowflake-Labs/snowflake-demo-notebooks/blob/main/Ingest%20Public%20JSON/Ingest%20Public%20JSON.ipynb)のチュートリアルをご確認ください。 

In [ ]:
-- Teardown stage created as part of this tutorial
DROP STAGE PERMANENT_STAGE;